# Tomograf stożkowy - Julia Miśta, Tymon Błaszkowski

## Import bibliotek

In [12]:
import math
import skimage
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.animation as animation
from IPython.display import HTML
from save_dicom import save_as_dicom

## Wyświelanie animacji sinogramu i zrekonstruowanego obrazu

In [2]:
def animate_reconstruction(steps, step_freq):
    fig, ax = plt.subplots(figsize=(5, 5))
    img = ax.imshow(steps[0], cmap='gray')
    title = ax.set_title("Rekonstrukcja")

    ax.axis('off')

    def update(frame):
        img.set_data(steps[frame])
        title.set_text(f"Po {frame * step_freq} rzutach")
        return [img]

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(steps),
        interval=200,
        blit=True,
        repeat = False
    )

    html = HTML(ani.to_jshtml())
    plt.close(fig)

    display(html)

def animate_sinogram(steps, step_freq):
    fig, ax = plt.subplots(figsize=(5, 5))
    img = ax.imshow(steps[0], cmap='gray', aspect='auto')
    title = ax.set_title("Sinogram")

    def update(frame):
        img.set_data(steps[frame])
        title.set_text(f"Po {frame * step_freq} rzutach")
        return [img]

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(steps),
        interval=200,
        blit=True,
        repeat = False
    )

    html = HTML(ani.to_jshtml())
    plt.close(fig)

    display(html)

## Filtrowanie wyjściowego obrazu

In [3]:
def ramp_filter(sinogram):
    num_angles, n_det = sinogram.shape
    freq = np.fft.fftfreq(n_det).reshape(-1, 1)
    filtr = np.abs(freq) 
    filtered_sinogram = np.zeros_like(sinogram)
    for i in range(num_angles):
        line_fft = np.fft.fft(sinogram[i, :])
        filtered_line = line_fft * filtr.ravel()
        filtered_sinogram[i, :] = np.real(np.fft.ifft(filtered_line))
    return filtered_sinogram

## Funkcja *detectorCords*
Funkcja służąca do wyliczenia lokalizacji - współrzędnych x,y danego dektektora.  

Wejście:  
 * a - wartość kąta na którym znajduje się dany emiter poruszając się po okręgu  
 * i - numer emitera  
 * phi - rozpietość kątowa między emiterami  
 * r - promień trajektorii układu pomiarowego, określający stałą odległość emitera i detektorów od środka obrotu (środka obrazu)  
 
Wyjście:  
 * xd, yd - współrzędne detektora

In [4]:
def detectorCords(a,n,phi,r,i):

    if i == 0:
        xd = r * math.cos(a + math.pi - phi/2)
        yd = r * math.sin(a + math.pi - phi/2)
    elif i == n-1:
        xd = r * math.cos(a + math.pi + phi/2)
        yd = r * math.sin(a + math.pi + phi/2)
    else: #di
        xd = r * math.cos(a + math.pi - phi/2 + i * phi/(n-1))
        yd = r * math.sin(a + math.pi - phi/2 + i * phi/(n-1))

    return xd, yd

## Funkcja *get_pixels_on_line*
Funkcja służąca do wyznaczania współrzędnych pikseli leżących na odcinku łączącym emiter z detektorem.

Wejście:  
 * x1,y1 - współrzędne emitera
 * x2,y2 - współrzędne detektora 
 
Wyjście:  
 * rows - zawiera indeksy wierszy (y) wszystkich pikseli na linii
 * cols -  zawiera indeksy kolumn (x) wszystkich pikseli na linii

In [5]:
def get_pixels_on_line(x1, y1, x2, y2):
    start_y, start_x = int(round(y1)), int(round(x1))
    end_y, end_x = int(round(y2)), int(round(x2))
    rows, cols = skimage.draw.line_nd((start_y, start_x), (end_y, end_x), endpoint=True)
    return rows, cols

## Wczytywanie obrazu
Wczytanie pliku wejściowego, konwersja z formatu RGB na skalę szarości oraz ujednolicenie rozmiaru obrazu do 256x256 px. Na koniec wartości pikseli są przeskalowane do zakresu [0, 255].

In [35]:
img = skimage.io.imread("tomograf-obrazy/Shepp_logan.JPG")

if len(img.shape) == 3:
    img_grey = skimage.color.rgb2gray(img)
else:
    img_grey = img
img_grey = skimage.transform.resize(img_grey, (256, 256))
img_grey = (img_grey * 255).astype(np.uint8)

## Tworzenie sinogramu - *generate_sinogram*
Funkcja generate_sinogram symuluje proces skanowania obiektu poprzez obrót układu emiter-detektory o $360^\circ$. Dla każdego położenia kątowego oblicza ona średnią intensywność pikseli wzdłuż linii rzutu, tworząc wynikową macierz projekcji. Dodatkowo funkcja rejestruje stany pośrednie całego procesu, co pozwala na późniejszą animację powstawania sinogramu.

In [38]:
def generate_sinogram(delta_a, n, phi_deg):
    sinogram_steps = []
    curr_h, curr_w = img_grey.shape
    r = math.sqrt(curr_w**2 + curr_h**2) / 2 + 10 
    phi = math.radians(phi_deg)
    
    angles = np.arange(0, 360, delta_a)
    num_steps = len(angles)
    sinogram = np.zeros((num_steps, n))
    offset_x, offset_y = curr_w / 2, curr_h / 2

    num_snapshots = 20
    step_freq = max(1, len(angles) // num_snapshots)

    for view, current_angle in enumerate(angles):
        current_angle_rad = math.radians(current_angle)
        xe = r * math.cos(current_angle_rad)
        ye = r * math.sin(current_angle_rad)
        
        for D in range(n):
            xd, yd = detectorCords(current_angle_rad, n, phi, r, D)
            rows, cols = get_pixels_on_line(xe + offset_x, ye + offset_y, 
                                            xd + offset_x, yd + offset_y)
            rows = np.array(rows)
            cols = np.array(cols)
            mask = (rows >= 0) & (rows < curr_h) & (cols >= 0) & (cols < curr_w)
            pixels = img_grey[rows[mask], cols[mask]]
            
            if len(pixels) > 0:
                sinogram[view, D] = np.mean(pixels)

        if view % step_freq == 0 or view == len(angles) - 1:
            temp_sino = np.zeros_like(sinogram)
            temp_sino[:view+1, :] = sinogram[:view+1, :]
            sinogram_steps.append(temp_sino)

    return sinogram, sinogram_steps, step_freq, angles

## Rekonstrukcja obrazu z sinogramu - *reconstruction_process*
Funkcja reconstruction_process odtwarza obraz z sinogramu. Po nałożeniu filtru w celu wyostrzenia krawędzi, wartości rzutów są iteracyjnie nanoszone na macierz obrazu pod odpowiednimi kątami. Wynikowy obraz powstaje poprzez uśrednienie skumulowanych danych i skalowanie ich intensywności. Całość procesu jest rejestrowana w krokach, co pozwala na animację powstawania rekonstrukcji.

In [37]:
def reconstruction_process(sinogram, angles, n, phi, r, offset_x, offset_y, curr_h, curr_w, step_freq):

    filtered_sinogram = ramp_filter(sinogram)
    reconstruction = np.zeros((curr_h, curr_w))
    hits = np.zeros((curr_h, curr_w))
    reconstruction_steps = [] 
    
    num_steps = len(angles)

    for view, current_angle in enumerate(angles):
        current_angle_rad = math.radians(current_angle)
        xe = r * math.cos(current_angle_rad)
        ye = r * math.sin(current_angle_rad)
        
        for D in range(n):
            val = filtered_sinogram[view, D]
            xd, yd = detectorCords(current_angle_rad, n, phi, r, D)
            rows, cols = get_pixels_on_line(xe + offset_x, ye + offset_y, 
                                            xd + offset_x, yd + offset_y)
            
            mask = (rows >= 0) & (rows < curr_h) & (cols >= 0) & (cols < curr_w)
            reconstruction[rows[mask], cols[mask]] += val
            hits[rows[mask], cols[mask]] += 1

        if view % step_freq == 0 or view == len(angles) - 1:
            temp = np.divide(reconstruction, hits, out=np.zeros_like(reconstruction), where=hits!=0)
            reconstruction_steps.append(temp.copy())

    final = np.divide(reconstruction, hits, out=np.zeros_like(reconstruction), where=hits!=0)
    v_min, v_max = np.percentile(final, (5, 99))
    final_scaled = skimage.exposure.rescale_intensity(final, in_range=(v_min, v_max))
    steps_scaled = [skimage.exposure.rescale_intensity(s, in_range=(v_min, v_max)) for s in reconstruction_steps]
    
    return final_scaled, steps_scaled

## Wyświetlanie wyników - *display_results*

In [9]:
def display_results(img_grey, sinogram, reconstructed_plot, num_steps):
    fig, ax = plt.subplots(1, 3, figsize=(18, 6))
    ax[0].imshow(img_grey, cmap='gray')
    ax[0].set_title("Original Image")
    ax[1].imshow(sinogram, cmap='gray', aspect='auto')
    ax[1].set_title(f"Sinogram\n({num_steps} views)")
    ax[2].imshow(reconstructed_plot, cmap='gray')
    ax[2].set_title("Reconstruction")
    plt.show()

## Funkcja *run_simulation*
Funkcja run_simulation integruje cały proces symulacji. Odpowiada za konfigurację geometrii układu, wygenerowanie sinogramu oraz przeprowadzenie rekonstrukcji obrazu. Następnie zarządza prezentacją danych: wyświetla statyczne porównanie obrazów, eksportuje finalny wynik do formatu DICOM wraz z metadanymi pacjenta oraz uruchamia interaktywne animacje obrazujące przebieg całego badania.

In [33]:
def run_simulation(delta_a, n, phi_deg, name, patient_id, comments):
    h, w = img_grey.shape
    r = math.sqrt(w**2 + h**2) / 2 + 10
    phi_rad = math.radians(phi_deg)
    offset_x, offset_y = w / 2, h / 2
    
    sinogram, sino_steps, step_freq, angles = generate_sinogram(delta_a, n, phi_deg)
    
    reconstruction, reco_steps = reconstruction_process(sinogram, angles, n, phi_rad, r, 
        offset_x, offset_y, h, w, step_freq)
    
    display_results(img_grey, sinogram, reconstruction, len(angles))
    
    patient_data = {"PatientName": name, "PatientID": patient_id, "ImageComments": comments}
    final_dcm = (reconstruction * 255).astype(np.uint8)
    save_as_dicom("out.dcm", final_dcm, patient_data)
    
    animate_sinogram(sino_steps, step_freq)
    animate_reconstruction(reco_steps, step_freq)


## Graficzny interfejs użytkownika

In [ ]:
style = {'description_width': '120px'}
interact_manual(
    run_simulation,
    delta_a = widgets.FloatSlider(value=2.0, min=0.5, max=10.0, step=0.5, description='Krok (Δα):', style=style),
    n = widgets.IntSlider(value=180, min=10, max=360, step=10, description='Detektory (n):', style=style),
    phi_deg = widgets.IntSlider(value=180, min=10, max=270, step=5, description='Rozpiętość (φ°):', style=style),
    name = widgets.Text(value='Jan Kowalski', description='Pacjent:', style=style),
    patient_id = widgets.Text(value='12345', description='ID Pacjenta:', style=style),
    comments = widgets.Textarea(value='Skan testowy', description='Komentarze:', style=style)
);

interactive(children=(FloatSlider(value=2.0, description='Krok (Δα):', max=10.0, min=0.5, step=0.5, style=Slid…

##     